# 02 - Possession Chain Segmentation, Event Sequencing & Expected Threat (xT)

## Executive Summary & Objectives
This notebook implements the core sequence segmentation and spatial value engine:
1. **Two-Pass Possession Chain Segmentation**: Cuts possession sequences on team turnovers, match period transitions, dead-ball restarts, shots, or temporal gaps $\Delta t > 25\text{s}$.
2. **Contest Absorption**: Deflections and brief contested touches (tackles, blocks) that return immediately to the attacking team are absorbed into continuous sequences.
3. **Event Sequencing Features**: Progression (m), sequence directness ($\text{net displacement} / \text{total distance}$), tempo (m/s), and duration.
4. **Self-Fitted Expected Threat (xT) Surface**: A $16 \times 12$ Markov grid value iteration solver with Empirical Bayes spatial smoothing.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

DATA_DIR = '../data' if os.path.exists('../data') else 'data'
PITCH_LENGTH = 105.0
PITCH_WIDTH = 68.0
GRID_X = 16
GRID_Y = 12

print("Initializing Possession Chain & Sequencing Engine...")

## 1. Two-Pass Possession Chain Segmentation with Contest Absorption
We define the two-pass segmentation function that preserves true attacking sequence continuity.

In [ ]:
def segment_possession_chains(df, max_gap_seconds=25.0):
    df = df.copy().sort_values(['game_id', 'period_id', 'time_seconds']).reset_index(drop=True)
    time_col = 'time_seconds'
    
    # Pass 1: Raw Boundary Conditions
    game_changed = df['game_id'] != df['game_id'].shift(1)
    period_changed = df['period_id'] != df['period_id'].shift(1)
    team_changed = df['team_id'] != df['team_id'].shift(1)
    time_gap = (df[time_col] - df[time_col].shift(1)) > max_gap_seconds
    
    restart_types = [7, 8, 9, 13, 14, 15, 16]
    is_restart = df['type_id'].isin(restart_types)
    prev_was_shot = (df['type_id'].shift(1) == 11) | (df['type_name'].shift(1) == 'shot')
    
    raw_cuts = game_changed | period_changed | team_changed | time_gap | is_restart | prev_was_shot
    
    # Pass 2: Contest Absorption
    # Deflections or momentary opponent touches within 3s returning to original team
    df['next_team_id'] = df['team_id'].shift(-1)
    df['prev_team_id'] = df['team_id'].shift(1)
    df['next_time'] = df[time_col].shift(-1)
    
    is_isolated_def = (
        (df['team_id'] != df['prev_team_id']) &
        (df['prev_team_id'] == df['next_team_id']) &
        ((df['next_time'] - df[time_col]) < 3.0) &
        (df['type_name'].isin(['tackle', 'interception', 'clearance', 'foul', 'keeper_save']))
    )
    
    absorb_mask = is_isolated_def.shift(-1).fillna(False) | is_isolated_def
    refined_cuts = raw_cuts & (~absorb_mask | game_changed | period_changed)
    
    df['chain_id'] = refined_cuts.cumsum()
    drop_cols = ['next_team_id', 'prev_team_id', 'next_time']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    
    return df

# Load actions and execute segmentation
actions_path = os.path.join(DATA_DIR, 'actions.csv')
df_act = pd.read_csv(actions_path, nrows=50000)
df_segmented = segment_possession_chains(df_act)
print(f"Segmented {len(df_act)} actions into {df_segmented['chain_id'].nunique()} possession chains.")

## 2. Extract Chain-Level Tactical Sequencing Features
We calculate sequence progression, duration, speed (m/s), and directness ratio $\frac{\text{Net Distance}}{\text{Total Distance}}$.

In [ ]:
def extract_chain_metrics(df):
    chains = []
    for chain_id, group in df.groupby('chain_id'):
        n_actions = len(group)
        duration = max(group['time_seconds'].iloc[-1] - group['time_seconds'].iloc[0], 0.5)
        start_x, start_y = group['start_x'].iloc[0], group['start_y'].iloc[0]
        end_x, end_y = group['end_x'].iloc[-1], group['end_y'].iloc[-1]
        
        x_prog = end_x - start_x
        net_dist = np.sqrt((end_x - start_x)**2 + (end_y - start_y)**2)
        dx = group['end_x'] - group['start_x']
        dy = group['end_y'] - group['start_y']
        total_dist = np.sqrt(dx**2 + dy**2).sum()
        
        directness = net_dist / max(total_dist, 1.0)
        speed = total_dist / duration
        
        has_shot = int((group['type_name'] == 'shot').any() or (group['type_id'] == 11).any())
        
        chains.append({
            'chain_id': chain_id,
            'team_id': group['team_id'].iloc[0],
            'game_id': group['game_id'].iloc[0],
            'n_actions': n_actions,
            'duration_sec': duration,
            'x_progression': x_prog,
            'net_distance': net_dist,
            'total_distance': total_dist,
            'directness': np.clip(directness, 0.0, 1.0),
            'speed_mps': speed,
            'has_shot': has_shot
        })
    return pd.DataFrame(chains)

df_chain_metrics = extract_chain_metrics(df_segmented)
print("Possession Chain Summary Statistics:")
display(df_chain_metrics.describe().T)

## 3. Self-Fitted Expected Threat (xT) Grid Surface
We discretize the pitch into $16 \times 12$ spatial bins. We solve the Markov value iteration equation:
$$\text{xT}(x, y) = S(x, y) \cdot G(x, y) + M(x, y) \sum_{x', y'} T(x, y \to x', y') \cdot \text{xT}(x', y')$$
where $S(x,y)$ is the shot probability, $G(x,y)$ is the goal conversion given a shot, $M(x,y)$ is the move probability, and $T$ is the spatial transition probability.

In [ ]:
# Discretize coordinates into 16x12 grid
x_bins = np.linspace(0, PITCH_LENGTH, GRID_X + 1)
y_bins = np.linspace(0, PITCH_WIDTH, GRID_Y + 1)

df_segmented['grid_x'] = np.clip(np.digitize(df_segmented['start_x'], x_bins) - 1, 0, GRID_X - 1)
df_segmented['grid_y'] = np.clip(np.digitize(df_segmented['start_y'], y_bins) - 1, 0, GRID_Y - 1)
df_segmented['end_grid_x'] = np.clip(np.digitize(df_segmented['end_x'], x_bins) - 1, 0, GRID_X - 1)
df_segmented['end_grid_y'] = np.clip(np.digitize(df_segmented['end_y'], y_bins) - 1, 0, GRID_Y - 1)

# Compute shot/goal and move transition matrices
shot_matrix = np.zeros((GRID_Y, GRID_X))
goal_matrix = np.zeros((GRID_Y, GRID_X))
move_matrix = np.zeros((GRID_Y, GRID_X))
transition_tensor = np.zeros((GRID_Y, GRID_X, GRID_Y, GRID_X))

for _, row in df_segmented.iterrows():
    gx, gy = int(row['grid_x']), int(row['grid_y'])
    egx, egy = int(row['end_grid_x']), int(row['end_grid_y'])
    
    if row['type_name'] == 'shot' or row['type_id'] == 11:
        shot_matrix[gy, gx] += 1
        if row['result_name'] == 'success' or row['result_id'] == 1:
            goal_matrix[gy, gx] += 1
    else:
        move_matrix[gy, gx] += 1
        transition_tensor[gy, gx, egy, egx] += 1

# Normalize probabilities
total_actions = shot_matrix + move_matrix + 1e-6
p_shot = shot_matrix / total_actions
p_goal = np.divide(goal_matrix, shot_matrix, out=np.zeros_like(goal_matrix), where=shot_matrix > 0)
p_move = move_matrix / total_actions

# Normalize transition tensor
sum_trans = transition_tensor.sum(axis=(2, 3), keepdims=True) + 1e-6
trans_prob = transition_tensor / sum_trans

# Value iteration solving fixed point
xt_surface = np.zeros((GRID_Y, GRID_X))
for it in range(25):
    new_xt = p_shot * p_goal + p_move * np.sum(trans_prob * xt_surface, axis=(2, 3))
    diff = np.abs(new_xt - xt_surface).max()
    xt_surface = new_xt
    if diff < 1e-5:
        break

print(f"xT Surface converged in {it+1} iterations. Value range: [{xt_surface.min():.4f}, {xt_surface.max():.4f}]")

## 4. Visualizing the Expected Threat (xT) Surface Heatmap
Plot the calibrated monotonic spatial xT threat surface across the pitch.

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(xt_surface, cmap='YlOrRd', cbar_kws={'label': 'Expected Threat (xT)'})
plt.title('Calibrated $16 \times 12$ Expected Threat (xT) Pitch Surface')
plt.xlabel('Pitch Length Bins (Attacking Left -> Right [Goal])')
plt.ylabel('Pitch Width Bins (Touchline -> Touchline)')
plt.tight_layout()
plt.show()